# IT-Salary-Predictor: Preprocessing & Eksploracyjna Analiza Danych (EDA)
**Autor:** Osoba 2 (Analityk Danych)  
**Projekt:** Analiza i predykcja stawek rynkowych w branzy IT  
**Etapy sylabusa:** 
- **Etap 2:** Pozyskanie i przygotowanie danych (czyszczenie, obsluga brakow, EDA)
- **Etap 4:** Wykorzystanie danych z relacyjnej bazy danych (PostgreSQL)

---
### Cele analityczne:
1. Pobranie surowych rekordow ofert pracy z tabeli 
aw_offers w bazie PostgreSQL.
2. Przetransformowanie i ustrukturyzowanie danych JSON do formatu relacyjnego/tabelarycznego.
3. Normalizacja widelek placowych (przeliczenie B2B i UoP do wspolnego mianownika PLN/miesiecznie).
4. Standaryzacja nazw technologii (np. React.js -> React, python -> Python).
5. Kompleksowa analiza eksploracyjna rynku:
   - Rozklad zarobkow wg doswiadczenia (Junior, Mid, Senior, Lead).
   - Porownanie zarobkow w podziale na formy zatrudnienia (B2B vs UoP).
   - Ranking najpopularniejszych oraz najlepiej wycenianych technologii.
   - Analiza geograficzna i wplyw pracy zdalnej.
6. Zapis wyczyszczonego zbioru analitycznego do tabeli cleaned_offers w bazie PostgreSQL (dla Inzyniera ML).


In [ ]:
import sys
import os
from pathlib import Path

# Umozliwienie importu modulow z katalogu src/
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.pipeline.db import fetch_raw_offers_df, save_cleaned_offers_df, test_connection
from src.pipeline.cleaner import transform_raw_offers_dataframe, normalize_skill

# Konfiguracja stylu wykresow
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('[OK] Wszystkie biblioteki zaimportowane pomyslnie.')
print('Status polaczenia z PostgreSQL:', 'POLACZONO' if test_connection() else 'BRAK POLACZENIA (uzyty zostanie plik zrzutu data/raw/offers_latest.json)')


## 1. Pobranie surowych danych z bazy PostgreSQL
Odpytujemy tabele 
aw_offers, w ktorej Inzynier Danych zapisal najnowszy zrzut ofert w formacie JSONB.


In [ ]:
df_raw = fetch_raw_offers_df()
print(f'Pobrano {len(df_raw):,} surowych rekordow ofert.')
df_raw.head()


## 2. Czyszczenie i transformacja danych (ETL)
Wywolujemy modul 	ransform_raw_offers_dataframe, ktory:
- Wyciaga atrybuty podstawowe (	itle, company_name, category, seniority, city, workplace_type).
- Przelicza stawki B2B i UoP do wartosci miesiecznych w PLN (salary_b2b_avg, salary_uop_avg).
- Tworzy glowna zmienna celu (	arget_salary_avg).
- Normalizuje technologie w skills.


In [ ]:
df_clean = transform_raw_offers_dataframe(df_raw)
print('Rozmiar wyczyszczonej tabeli:', df_clean.shape)
df_clean.head(5)


## 3. Analiza jakosci i brakujacych danych
Sprawdzamy, jaki odsetek ofert posiada jawne wideelki placowe (jest to kluczowe zagadnienie analityczne na polskim rynku IT).


In [ ]:
total = len(df_clean)
with_salary = df_clean['target_salary_avg'].notna().sum()
pct = (with_salary / total) * 100

print(f'Wszystkie oferty: {total:,}')
print(f'Oferty z podanymi zarobkami: {with_salary:,} ({pct:.1f}%)')
print(f'Oferty z ukrytymi zarobkami: {total - with_salary:,} ({100 - pct:.1f}%)')

# DataFrame zawierajacy wylacznie oferty z wyplata (do analizy zarobkow i modelowania)
df_salaries = df_clean[df_clean['target_salary_avg'].notna()].copy()


## 4. Eksploracyjna Analiza Danych (EDA)

### 4.1. Wynagrodzenia a poziom doswiadczenia (Seniority)
Jak rosna zarobki programistow w zaleznosci od stazu (Junior -> Mid -> Senior)?


In [ ]:
seniority_order = ['junior', 'mid', 'senior', 'c-level']
valid_seniority = df_salaries[df_salaries['seniority'].isin(seniority_order)]

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=valid_seniority,
    x='seniority',
    y='target_salary_avg',
    order=seniority_order,
    palette='Blues_d'
)
plt.title('Rozklad wynagrodzen w zaleznosci od poziomu doswiadczenia (PLN/miesiecznie)', fontsize=14, fontweight='bold')
plt.xlabel('Poziom doswiadczenia')
plt.ylabel('Srednia stawka miesieczna (PLN)')
plt.show()

# Tabela statystyk opisowych
stats_sen = valid_seniority.groupby('seniority')['target_salary_avg'].agg(['count', 'median', 'mean', 'std']).reindex(seniority_order)
stats_sen


### 4.2. Ranking najczesciej poszukiwanych technologii
Ekstrahujemy wszystkie znormalizowane technologie i sprawdzamy, jakie kompetencje sa najbardziej poszukiwane przez pracodawcow.


In [ ]:
all_skills = df_clean['skills'].explode().dropna()
top_skills = all_skills.value_counts().head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_skills.values, y=top_skills.index, palette='viridis')
plt.title('Top 15 najczesciej wymaganych technologii w ogloszeniach IT', fontsize=14, fontweight='bold')
plt.xlabel('Liczba ofert pracy')
plt.ylabel('Technologia')
plt.show()


### 4.3. Porownanie stawek: B2B vs Umowa o Prace (UoP)
Czy stawki na B2B sa wyraznie wyzsze niz na etacie?


In [ ]:
contract_data = df_salaries[df_salaries['primary_contract_type'].isin(['b2b', 'uop'])]

plt.figure(figsize=(10, 5))
sns.histplot(
    data=contract_data,
    x='target_salary_avg',
    hue='primary_contract_type',
    kde=True,
    bins=40,
    palette={'b2b': '#2ecc71', 'uop': '#3498db'}
)
plt.title('Porownanie rozkladu wynagrodzen: B2B vs UoP', fontsize=14, fontweight='bold')
plt.xlabel('Wynagrodzenie miesieczne (PLN)')
plt.ylabel('Liczba ofert')
plt.show()


## 5. Zapisanie wyczyszczonej tabeli do PostgreSQL
Zapisujemy gotowa tabele cleaned_offers do relacyjnej bazy danych.  
Z tej tabeli bezposrednio korzystac beda:
1. **Osoba 3 (Inzynier ML)** – do treningu modeli regresji (XGBoost, Random Forest).
2. **Osoba 4 (Frontendowiec)** – do serwowania statystyk rynkowych w kalkulatorze.


In [ ]:
# Zapis do PostgreSQL (oraz lokalna kopia CSV/Parquet)
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/cleaned_offers.csv', index=False)
print('[OK] Zapisano lokalny plik data/processed/cleaned_offers.csv')

if test_connection():
    save_cleaned_offers_df(df_clean, if_exists='replace')
    print('[OK] Pomyslnie zapisano tabele cleaned_offers w PostgreSQL!')
else:
    print('[INFO] PostgreSQL nie jest obecnie wlaczony. Dane zapisano w pliku processed/cleaned_offers.csv.')
